# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/supriya-006/FlyRank_Assignment/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

1. Finding: the paper reports a large lift from the baseline to the learned model at precision@50. My methodology question is this: where does the label come from, and does the validation design preserve the real deployment boundary? A careful reviewer would ask whether the target is defined from a future window, whether the split respects client clustering, and whether any feature family partly encodes the label itself. In other words, the claim is stronger only if the label is measured before the decision and the test set is not contaminated by repeated client patterns.

2. Finding: the baseline is described as a transparent and fair comparison. My methodology question is whether the baseline uses the same split, the same feature timing, and the same decision rule as the learned model. If the baseline is built on a different window or with product-derived signals, then the comparison is not apples-to-apples. A fair audit asks whether the model is beating the same decision problem under the same constraints, not just a more convenient proxy.

In [1]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
for candidate in [ROOT, ROOT.parent, ROOT.parent.parent]:
    if (candidate / "data").exists() and (candidate / "scripts").exists():
        ROOT = candidate
        break

if (ROOT / "scripts").exists():
    sys.path.insert(0, str(ROOT / "scripts"))

print("Repo root:", ROOT)

df = pd.read_csv(ROOT / "data" / "processed" / "refresh_feature_vector.csv")
print("Rows:", len(df))
print("Positive rate:", round(df["is_declining_label"].mean(), 3))
print("Clients:", df["client_id"].nunique())
print("Label-related columns in the dataset:", [c for c in ["trend_pct", "trend_direction", "is_declining_label", "content_id", "client_id"] if c in df.columns])

Repo root: /home/supriya-devkota/Desktop/FlyRank_Assignment
Rows: 30000
Positive rate: 0.542
Clients: 32
Label-related columns in the dataset: ['trend_pct', 'trend_direction', 'is_declining_label', 'content_id', 'client_id']


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

The honest check is a client-held-out split. A random row split often overstates model quality because pages from the same client share patterns; the client-grouped split better reflects the real editorial decision boundary. I therefore reran the same model family under both a random row split and a grouped client split. The gap between those numbers is itself evidence about how much memorization may have been happening under the weaker split.

In [2]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from ml_utils import MODEL_CATEGORICAL_FEATURES, MODEL_NUMERIC_FEATURES, precision_at_k


def summary_frame(y_true, scores, label):
    pred = (scores >= 0.5).astype(int)
    return {
        "method": label,
        "accuracy": round(accuracy_score(y_true, pred), 4),
        "precision": round(precision_score(y_true, pred, zero_division=0), 4),
        "recall": round(recall_score(y_true, pred, zero_division=0), 4),
        "f1": round(f1_score(y_true, pred, zero_division=0), 4),
        "roc_auc": round(roc_auc_score(y_true, scores), 4),
        "precision_at_20": round(precision_at_k(y_true, scores, 20), 4),
        "precision_at_50": round(precision_at_k(y_true, scores, 50), 4),
        "precision_at_100": round(precision_at_k(y_true, scores, 100), 4),
    }


def make_model(estimator, X_df):
    numeric_features = [c for c in MODEL_NUMERIC_FEATURES if c in X_df.columns]
    categorical_features = [c for c in MODEL_CATEGORICAL_FEATURES if c in X_df.columns]
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]),
                numeric_features,
            ),
            (
                "cat",
                Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]),
                categorical_features,
            ),
        ]
    )
    return Pipeline([("preprocess", preprocessor), ("model", estimator)])


def evaluate_split(train_df, test_df, model_name, estimator):
    numeric_features = [c for c in MODEL_NUMERIC_FEATURES if c in train_df.columns]
    categorical_features = [c for c in MODEL_CATEGORICAL_FEATURES if c in train_df.columns]
    X_train = train_df[numeric_features + categorical_features].copy()
    X_test = test_df[numeric_features + categorical_features].copy()
    y_train = train_df["is_declining_label"].astype(int)
    y_test = test_df["is_declining_label"].astype(int)

    pipeline = make_model(estimator, train_df)
    pipeline.fit(X_train, y_train)
    scores = pipeline.predict_proba(X_test)[:, 1]
    return summary_frame(y_test.to_numpy(), scores, model_name)


df = pd.read_csv(ROOT / "data" / "processed" / "refresh_feature_vector.csv")
baseline_df = pd.read_csv(ROOT / "data" / "processed" / "baseline_refresh_queue.csv")

random_idx = df.sample(frac=1, random_state=42).index
split_point = int(len(df) * 0.8)
random_train = df.loc[random_idx[:split_point]].copy()
random_test = df.loc[random_idx[split_point:]].copy()

GSS = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(GSS.split(df, groups=df["client_id"]))
grouped_train = df.iloc[train_idx].copy()
grouped_test = df.iloc[test_idx].copy()

results = []
for split_name, train_df, test_df in [
    ("Random row split", random_train, random_test),
    ("Grouped client split", grouped_train, grouped_test),
]:
    for model_name, estimator in [
        ("Logistic Regression", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42)),
        ("Random Forest", RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=25, class_weight="balanced_subsample", random_state=42)),
    ]:
        result = evaluate_split(train_df, test_df, model_name, estimator)
        result["split"] = split_name
        results.append(result)

    baseline_lookup = baseline_df.set_index("content_id")["baseline_refresh_score"]
    baseline_scores = test_df["content_id"].map(baseline_lookup).fillna(0).to_numpy()
    baseline_summary = summary_frame(test_df["is_declining_label"].astype(int).to_numpy(), baseline_scores, "Baseline")
    baseline_summary["split"] = split_name
    results.append(baseline_summary)

comparison = pd.DataFrame(results)
print(comparison[["split", "method", "precision_at_50", "roc_auc", "f1"]].sort_values(["split", "precision_at_50"], ascending=[True, False]).to_string(index=False))

               split              method  precision_at_50  roc_auc     f1
Grouped client split Logistic Regression             0.72   0.6158 0.6058
Grouped client split       Random Forest             0.58   0.6090 0.5936
Grouped client split            Baseline             0.32   0.4979 0.3222
    Random row split       Random Forest             0.96   0.7727 0.7276
    Random row split Logistic Regression             0.88   0.7309 0.6879
    Random row split            Baseline             0.42   0.5802 0.5122


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The main leakage check here is straightforward: no label-derived or sibling columns are included in the features. The model uses search, freshness, engagement, and position variables that are knowable before the decision point, while explicitly excluding trend-derived columns and ID-like fields from training. The audit also checks whether a suspicious feature can inflate the metric dramatically; if it does, that is evidence of leakage rather than real skill.

In [3]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from ml_utils import MODEL_CATEGORICAL_FEATURES, MODEL_NUMERIC_FEATURES, precision_at_k


def make_preprocessor(feature_columns):
    numeric_features = [c for c in MODEL_NUMERIC_FEATURES if c in feature_columns]
    categorical_features = [c for c in MODEL_CATEGORICAL_FEATURES if c in feature_columns]
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_features),
            ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
        ]
    )


df = pd.read_csv(ROOT / "data" / "processed" / "refresh_feature_vector.csv")
feature_candidates = [c for c in MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES if c in df.columns]
leak_suspects = ["trend_pct", "trend_direction"]
print("Model feature count:", len(feature_candidates))
print("Label-derived suspects still present in feature set:", [c for c in leak_suspects if c in feature_candidates])
print("Excluded suspects:", [c for c in leak_suspects if c in df.columns and c not in feature_candidates])

if "trend_pct" in df.columns:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
    train_df = df.iloc[train_idx].copy()
    test_df = df.iloc[test_idx].copy()

    base_features = [c for c in feature_candidates if c != "trend_pct"]
    X_train = train_df[base_features].copy()
    X_test = test_df[base_features].copy()
    y_train = train_df["is_declining_label"].astype(int)
    y_test = test_df["is_declining_label"].astype(int)

    model = Pipeline([
        ("preprocess", make_preprocessor(base_features)),
        ("model", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42)),
    ])
    model.fit(X_train, y_train)
    base_score = precision_at_k(y_test.to_numpy(), model.predict_proba(X_test)[:, 1], 50)

    leak_features = base_features + ["trend_pct"]
    X_train_leak = train_df[leak_features].copy()
    X_test_leak = test_df[leak_features].copy()

    model_leak = Pipeline([
        ("preprocess", make_preprocessor(leak_features)),
        ("model", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42)),
    ])
    model_leak.fit(X_train_leak, y_train)
    leak_score = precision_at_k(y_test.to_numpy(), model_leak.predict_proba(X_test_leak)[:, 1], 50)

    print("Baseline honest precision@50 without trend_pct:", round(base_score, 4))
    print("Precision@50 with trend_pct added back as a leak:", round(leak_score, 4))

# Importance sanity check on the honest model
numeric_features = [c for c in MODEL_NUMERIC_FEATURES if c in df.columns]
categorical_features = [c for c in MODEL_CATEGORICAL_FEATURES if c in df.columns]
train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(df, groups=df["client_id"]))
train_df = df.iloc[train_idx].copy()

X_train = train_df[numeric_features + categorical_features].copy()
y_train = train_df["is_declining_label"].astype(int)

model = Pipeline([
    ("preprocess", make_preprocessor(numeric_features + categorical_features)),
    ("model", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42)),
])
model.fit(X_train, y_train)
feature_names = model.named_steps["preprocess"].get_feature_names_out()
importance = pd.DataFrame({
    "feature": feature_names,
    "importance": model.named_steps["model"].coef_[0],
}).sort_values("importance", ascending=False).head(10)
print("\nTop honest-model feature weights (top 10):")
print(importance.to_string(index=False))

Model feature count: 26
Label-derived suspects still present in feature set: []
Excluded suspects: ['trend_pct', 'trend_direction']
Baseline honest precision@50 without trend_pct: 0.72
Precision@50 with trend_pct added back as a leak: 0.72

Top honest-model feature weights (top 10):
                          feature  importance
         num__log_impressions_90d    1.425272
         cat__impression_tier_low    0.725328
   cat__word_count_tier_1000-2000    0.615490
         cat__main_intent_unknown    0.533893
                  num__word_count    0.508241
cat__content_type_keyword article    0.451658
         cat__freshness_tier_0-30    0.269560
       cat__freshness_tier_91-180    0.234515
      cat__position_tier_striking    0.196224
                 num__scroll_rate    0.181650


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original claim: "The model clearly improves refresh prioritization and should be trusted to decide which pages to review first." Honest rewrite: "On the observed client-held-out evaluation set, the logistic regression achieved precision@50 of 0.72 versus 0.32 for the baseline. This suggests a directional lift in editorial prioritization quality for this dataset and split, but it is not proof of causal impact on business outcomes or universal generalization beyond the measured setting."

In [4]:
safe_claim = (
    "On the observed client-held-out evaluation set, the logistic regression achieved "
    "precision@50 of 0.72 versus 0.32 for the baseline. This suggests a directional lift "
    "in editorial prioritization quality for this dataset and split, but it is not proof "
    "of causal impact on business outcomes or universal generalization beyond the measured "
    "setting."
)
print(safe_claim)

On the observed client-held-out evaluation set, the logistic regression achieved precision@50 of 0.72 versus 0.32 for the baseline. This suggests a directional lift in editorial prioritization quality for this dataset and split, but it is not proof of causal impact on business outcomes or universal generalization beyond the measured setting.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.